> ## SOLUTION / ANSWER KEY &mdash; Lab 8.11
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-11-assemble-chatbot.ipynb`](../lab-11-assemble-chatbot.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 8.11 &mdash; Assemble the Customer-Service Chatbot

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 35 min &nbsp;|&nbsp; **Day 4 &middot; Module 8 &mdash; Multi-Agent Collaboration &amp; Decision Making**

### What you'll do
- Build billing & tech specialists as real create_agent agents
- Route a message to only the matching specialists
- Synthesise one reply, flagged needs_approval for the refund

> **How this lab works (experiential flow):** read the **Concept**, run the **Demo** to see it work, then complete **Your Turn** by replacing every `___` placeholder. Run the **grader** cell at the end &mdash; it prints `[PASS]` / `[FAIL]` / `[TODO]` and a final `Score`. Aim for a full score.

> **Framework note:** these labs use the **real** LangChain (`langchain`, `langchain-core`, `langchain-ollama`). The **graded** cells assert only on the deterministic parts you build &mdash; routing, synthesis, tool wiring, agent structure &mdash; and never call an LLM, so the lab always verifies offline. Cells marked **Optional &mdash; run it for real** call a live local model (`ollama run llama3.2:1b`, or Groq) and self-skip if none is reachable. You are building the **multi-agent customer-service chatbot** &mdash; the client's Lab 4.2.

**Reference:** [Module 8 slides &mdash; The multi-agent customer-service chatbot](../../presentation/day4-module8-multi-agent-collaboration.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 8 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, socket
WORK = "/tmp/biaa-lab-08-11"
os.makedirs(WORK, exist_ok=True)
def ollama_up(host="127.0.0.1", port=11434):
    """True if a local Ollama server is listening -- the optional live cells self-skip when it isn't."""
    try:
        with socket.create_connection((host, port), timeout=1):
            return True
    except OSError:
        return False
print("Working dir:", WORK, "| Ollama reachable:", ollama_up())

## Concept
Now assemble the chatbot from Modules 6&ndash;7 pieces (deck slides 15&ndash;17): each specialist is a
**`create_agent`** with its **own** small tool set (a `CompiledStateGraph`). The **supervisor** routes
the message to only the matching specialists; their findings are **synthesised** into one reply. Because
a refund is **irreversible**, the reply is flagged **`needs_approval`** &mdash; draft-not-send, now at
the team level. The routing, synthesis and refund gate are deterministic (graded); the specialists run
in the optional cell.

In [ ]:
from langchain_core.tools import tool

# The chatbot's context sources: invoices (order 4471 has a DUPLICATE charge) and known issues.
INVOICES = {
    "4471": [{"amount": 50, "date": "Jul 01"}, {"amount": 50, "date": "Jul 01"}],
    "5090": [{"amount": 30, "date": "Jul 02"}],
}
KNOWN_ISSUES = {
    "crash": {"bug": "BUG-231", "fix": "update to v4.2"},
    "login": {"bug": "BUG-118", "fix": "reset your password"},
}

from langchain_core.tools import tool

@tool
def lookup_invoice(order_id: str) -> list:
    """Look up the charges on an order by id."""
    return INVOICES.get(order_id, [])
@tool
def known_issues(symptom: str) -> dict:
    """Look up a known technical issue by symptom keyword."""
    for k, v in KNOWN_ISSUES.items():
        if k in symptom.lower():
            return v
    return {}
def synthesize(findings):
    return " ".join(findings[k] for k in sorted(findings)) if findings else "No findings."
print("tools & synthesise ready:", lookup_invoice.name, "&", known_issues.name)

## Your Turn
Build each specialist with `create_agent`, complete `route` (which specialists a message needs), and
`assemble_reply` (synthesise + flag the refund for approval).

In [ ]:
from langchain_ollama import ChatOllama
from langchain.agents import create_agent

def build_specialist(tools, role):
    model = ChatOllama(model="llama3.2:1b")
    return create_agent(model, tools, system_prompt=f"You are the {role} specialist. Use only your tools.")

billing_agent = build_specialist([lookup_invoice], 'billing')
tech_agent    = build_specialist([known_issues], 'tech')

def route(message):
    m = message.lower()
    engaged = []
    if any(k in m for k in ("charg", "refund", "invoice", "billed")):
        engaged.append("billing")
    if any(k in m for k in ("crash", "bug", "login", "broken")):
        engaged.append("tech")
    return engaged

def assemble_reply(findings):
    # synthesise every specialist finding into ONE reply; a refund is irreversible -> needs a human
    return {"reply": synthesize(findings), "status": "needs_approval", "agents": sorted(findings)}

try:
    print('billing agent:', type(billing_agent).__name__)
    print('route (two-intent):', route('charged twice for 4471 and the app keeps crashing'))
    demo = assemble_reply({"billing": "duplicate charge, refund warranted", "tech": "matches BUG-231"})
    print('assembled    :', demo['status'], '| agents:', demo['agents'])
except Exception as e:
    print('Fill the blanks, then re-run.', type(e).__name__)

In [ ]:
# === Auto-grader: run after filling the blanks above ===
_results = []
def _rec(label, status, extra=""):
    _results.append(status); print(f"[{status}] {label}" + (f" -- {extra}" if extra else ""))
def expect(label, got, want):
    if got == "___" or got is None: _rec(label, "TODO")
    elif got == want: _rec(label, "PASS")
    else: _rec(label, "FAIL", f"got {got!r}")
def expect_true(label, fn):
    try: _rec(label, "PASS" if fn() else "FAIL")
    except Exception as e: _rec(label, "TODO", type(e).__name__)

expect_true("each specialist is a runnable CompiledStateGraph", lambda: type(billing_agent).__name__ == "CompiledStateGraph" and "tools" in set(tech_agent.nodes))
expect_true("a two-intent message engages BOTH specialists", lambda: route("charged twice for 4471 and the app keeps crashing") == ["billing", "tech"])
expect_true("a pure tech message engages only tech", lambda: route("the app keeps crashing") == ["tech"])
expect_true("a pure billing message engages only billing", lambda: route("I was charged twice") == ["billing"])
expect_true("synthesis merges both specialists' findings", lambda: (lambda r: "refund" in r.lower() and "bug-231" in r.lower())(assemble_reply({"billing": "refund warranted", "tech": "matches BUG-231"})["reply"]))
expect_true("the reply is needs_approval (refund is irreversible)", lambda: assemble_reply({"billing": "x"})["status"] == "needs_approval")

_p = _results.count("PASS")
print(f"\nScore: {_p}/{len(_results)}")
print("All checks passed -- lab complete!" if _p == len(_results) else "Keep going: fill the blanks marked ___ and re-run.")

## Optional &mdash; run it for real (not graded)
Run the real specialists behind the supervisor: route, run each matching agent, synthesise, gate the refund. This calls a **real** local model via `ChatOllama("llama3.2:1b")` &mdash; start it with
`ollama run llama3.2:1b` (or swap in `ChatGroq` with a `GROQ_API_KEY`). If none is reachable the cell
prints a note and moves on. **The graded cells above never call an LLM, so the lab always verifies offline.**
*(llama3.2:1b is tiny &mdash; tool-calling can be hit-or-miss; the point is to see a real invocation.)*

In [ ]:
def specialist_reply(agent, message):
    result = agent.invoke({"messages": [{"role": "user", "content": message}]}, config={"recursion_limit": 8})
    return result["messages"][-1].content
try:
    if ollama_up():
        msg = "I was charged twice for order 4471 and the app keeps crashing."
        agents = {"billing": billing_agent, "tech": tech_agent}
        findings = {name: specialist_reply(agents[name], msg) for name in route(msg)}
        out = assemble_reply(findings)
        print("agents:", out["agents"])
        print("status:", out["status"], "(refund is irreversible -> a human approves)")
        print("reply :", out["reply"])
    else:
        print("No Ollama reachable -- skipping the live run. The routing/synthesis/refund-gate above are what matters.")
except Exception as e:
    print("Live run skipped:", type(e).__name__)

---
### Key takeaway
A TEAM: a supervisor routes, specialists (each a create_agent with its own tools) gather, a synthesiser makes one reply -- and the refund waits for a human. Next: run it over a whole suite.

[&#8592; All Module 8 labs](./index.html) &nbsp;&middot;&nbsp; [Module 8 slides](../../presentation/day4-module8-multi-agent-collaboration.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>